# Computing Longitudinal Modularity

This notebook shows how to evaluate the quality of temporal community partitions using longitudinal modularity.

You can use this to:
- Compare different partitions
- Understand the quality metric
- Tune parameters for better results

In [ ]:
from lago import LexType, LinkStream, lago_modules, longitudinal_modularity

## Create Sample LinkStream

In [ ]:
def create_sample_linkstream():
    """Create a sample linkstream."""
    ls = LinkStream()
    ls.add_links([
        # Dense group 1
        (0, 1, 0), (1, 2, 0), (0, 2, 0),
        (0, 1, 1), (1, 2, 1),
        # Dense group 2
        (3, 4, 0), (4, 5, 0), (3, 5, 0),
        (3, 4, 1),
        # Bridge
        (2, 3, 1),
    ])
    return ls

ls = create_sample_linkstream()
print(f"Created linkstream with {ls.nb_nodes} nodes")

## 1. Basic Modularity Computation

Compute modularity for detected communities.

In [ ]:
communities = lago_modules(ls)

# Compute modularity
result = longitudinal_modularity(ls, communities)

print(f"Modularity: {result.value}")
print(f"Time penalty: {result.time_penalty}")
print(f"Modularity without penalty: {result.modularity_without_penalty}")
print(f"Expectation type: {result.lex.name}")

## 2. Comparing Expectation Types

Different longitudinal expectation types measure community quality differently:

- **MM (Mean-Membership):** Most flexible, general use
- **JM (Joint-Membership):** Favors stable communities
- **CM (Coexistence):** Based on node co-occurrence in time

In [ ]:
print("Modularity with different lex types:")
print("-" * 40)

for lex in [LexType.MM, LexType.JM, LexType.CM]:
    result = longitudinal_modularity(ls, communities, lex=lex)
    print(f"  {lex.name}: {result.value}")

## 3. Evaluating Manual Partitions

You can evaluate manually defined partitions to compare with LAGO's results.

In [ ]:
# Define communities manually as dict of (node, time) tuples

# Partition 1: All in one community
all_together = {
    0: {(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (5, 0), 
        (0, 1), (1, 1), (2, 1), (3, 1), (4, 1)}
}

# Partition 2: Two groups
two_groups = {
    0: {(0, 0), (1, 0), (2, 0), (0, 1), (1, 1), (2, 1)},
    1: {(3, 0), (4, 0), (5, 0), (3, 1), (4, 1)},
}

print("Comparing manual partitions:")
print("-" * 40)

for name, partition in [("All together", all_together), ("Two groups", two_groups)]:
    result = longitudinal_modularity(ls, partition)
    print(f"  {name}: {result.value:.4f}")

# LAGO's result
result = longitudinal_modularity(ls, communities)
print(f"  LAGO result: {result.value:.4f}")

## 4. Parameter Sensitivity

Understand how gamma and omega affect modularity.

### Gamma (resolution) effect

Higher gamma penalizes larger communities more.

In [ ]:
print("Gamma (resolution) effect:")
print("-" * 40)
for gamma in [0.0, 0.5, 1.0, 2.0]:
    result = longitudinal_modularity(ls, communities, gamma=gamma)
    print(f"  gamma={gamma}: modularity={result.value:.4f}")

### Omega (time smoothness) effect

Higher omega penalizes community changes over time more.

In [ ]:
print("Omega (time smoothness) effect:")
print("-" * 40)
for omega in [0.0, 1.0, 2.0, 5.0]:
    result = longitudinal_modularity(ls, communities, omega=omega)
    print(f"  omega={omega}: modularity={result.value:.4f}, penalty={result.time_penalty:.4f}")

## 5. Understanding Modularity Components

Longitudinal modularity has two main components:
- **Topological quality:** How well-connected are communities internally?
- **Time penalty:** How much do communities change over time?

In [ ]:
# Full modularity
full = longitudinal_modularity(ls, communities, gamma=1.0, omega=2.0)

# Without expectation term
no_expect = longitudinal_modularity(ls, communities, gamma=0.0, omega=2.0)

# Without time penalty
no_time = longitudinal_modularity(ls, communities, gamma=1.0, omega=0.0)

print("Modularity breakdown:")
print("-" * 40)
print(f"  Full modularity (γ=1, ω=2): {full.value:.4f}")
print(f"  Without expectation (γ=0):  {no_expect.value:.4f}")
print(f"  Without time penalty (ω=0): {no_time.value:.4f}")
print()
print("Components:")
print(f"  Internal edges contribution: ~{no_expect.modularity_without_penalty:.4f}")
print(f"  Expectation penalty: ~{no_expect.value - full.modularity_without_penalty:.4f}")
print(f"  Time penalty: {full.time_penalty:.4f}")

---

## Next Steps

Continue with:
- [05_visualization.ipynb](05_visualization.ipynb) - Create beautiful visualizations of your communities